In [1]:
!pip install ipdb -q
!pip install tqdm -q
!pip install sentencepiece -q
!pip install wandb -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 78.6 MB/s eta 0:00:00


In [4]:
!nvidia-smi

Wed Jun  3 20:18:41 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import os
import ipdb
from tqdm import tqdm
from datetime import datetime
import platform, shutil
import requests, zipfile, io

import torch
import torch.nn as nn
from torch.nn import functional as F

import sentencepiece as spm

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

torch.cuda.empty_cache()

In [3]:
files_url = "https://ideami.com/llm_train"

# Downloading proceeds if we detect that one of the key files to download is not present
if not os.path.exists(f"encoded_data.pt"):
    print("Downloading files using Python")
    response = requests.get(files_url)
    zipfile.ZipFile(io.BytesIO(response.content)).extractall(".")
else:
    print("you seem to have already downloaded the files. If you wish to re-download them, delete the encoded_data.pt file")

In [5]:
# PARAMETERS

## for collab batch_size = 32
batch_size = 8
context = 512
embed_size = 384
n_layers = 7
n_heads = 7
BIAS = True


# HYPERPARAMTERS
lr = 3e-4
dropout = 0.05
weight_decay = 0.01
grad_clip = 1.0

# TRAINING PARAMETERS
train_iters = 100000
eval_interval = 50
eval_iters = 10
compile = False

load_pretrained = False
checkpoint_dir = 'models/'
checkpoint_fn = 'latest.pt'
checkpoint_load_fn = 'latest.pt'
dtype = torch.bfloat16

# MODE
inference = False

# DEVICE
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device: You will be using: ",device)

device: You will be using:  cuda


In [6]:
# LOGGING
# wandb_log = False  # Set to True only when you switch to T4 GPU for the real run!
wandb_log = True
wandb_project = "llm4"
wandb_run_name = "llm4"

if wandb_log:
    import wandb
    wandb.init(project=wandb_project, name=wandb_run_name)

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice: 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.


wandb: Paste your API key and hit enter: ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: saleh-work2001 (saleh-work2001-north-south-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [7]:
with open('wiki.txt', 'r', encoding='utf-8') as f:
    text = f.read()
print(text[30000:30300])

terms.
For example, there are objects in two groups (as shown on the right). The objects are various shapes, where one group has 3 of them while the other has 2. When the two groups combine into one, the overall amount (sum) of the shapes become 5.

Vertical Addition

The animation above demonstrate


In [8]:
# TOKENIZER
sp = spm.SentencePieceProcessor(model_file='wiki_tokenizer.model')

vocab_size = sp.get_piece_size()
print("vocab_size: ",vocab_size)

# Create the encoding and decoding tokenizer functions
encode = lambda s: sp.Encode(s)
decode = lambda l: sp.Decode(l)


# Test that encoding and decoding are working well
print(decode(encode("Encoding Decoding functions ready")))

vocab_size:  4096
Encoding Decoding functions ready


In [9]:
# Tokenization of the dataset
if os.path.exists(f"encoded_data.pt"):
    # Load encoded data if saved it previously
    print("Loading saved encoded data")
    data = torch.load('encoded_data.pt')
else:
    # If still didn't encode and save the encoding, do it here
    print("Encoding data")
    data = torch.tensor(encode(text), dtype=torch.long)
    torch.save(data, 'encoded_data.pt')

Loading saved encoded data


In [10]:
data_size=len(data) # Get the size of the dataset

spl = int(0.9*data_size) # set the split at 90%-10%
train_data=data[:spl] # training data will be 90% of the dataset
val_data=data[spl:] # validation data will be 10% of the dataset
print(f'Total data: {data_size/1e6:.2f} Million | Training: {len(train_data)/1e6:.2f} Million | Validation: {len(val_data)/1e6:.2f} Million')

Total data: 59.21 Million | Training: 53.29 Million | Validation: 5.92 Million


In [11]:
def get_batch(split):
    # BS = Batch Size / SL = Sequence Length or context length
    data = train_data if split=="train" else val_data # Select the split
    inds = torch.randint(len(data)-context, (batch_size,)) # (BS)
    x = torch.stack([data[i: i+context] for i in inds]) # (BS,SL)
    y = torch.stack([data[i+1: i+context+1] for i in inds]) # (BS,SL)

    x,y = x.to(device), y.to(device)
    return x,y


# Uncomment to test your get_batch function
x,y=get_batch("train")
print(f"x.shape: {x.shape}")
print(f"y.shape: {y.shape}")
print(x[0][:10])
print(y[0][:10])

x.shape: torch.Size([8, 512])
y.shape: torch.Size([8, 512])
tensor([ 475, 4031, 4056, 4056, 4053,  307,  264,  340,  912, 3249],
       device='cuda:0')
tensor([4031, 4056, 4056, 4053,  307,  264,  340,  912, 3249,  938],
       device='cuda:0')


In [12]:
class GPT(nn.Module):
    def __init__(self):
        super().__init__()
        self.embeddings = nn.Embedding(vocab_size, embed_size)
        self.positions = nn.Embedding(context, embed_size)
        self.blocks = nn.Sequential(*[Block(n_heads) for _ in range(n_layers)])
        self.ln = nn.LayerNorm(embed_size)
        self.final_linear = nn.Linear(embed_size, vocab_size, bias=BIAS)
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, input, targets=None):
        loss = None
        BS, SL = input.shape
        emb = self.embeddings(input)
        pos = self.positions(torch.arange(SL, device=device))
        X = emb + pos
        X = self.blocks(X)
        X = self.ln(X)
        logits = self.final_linear(X)

        if targets is not None:
            BS, SL, VS = logits.shape
            logits = logits.view(BS*SL, VS)
            targets = targets.view(BS*SL)
            loss = F.cross_entropy(logits, targets)

            counts = logits.exp()
            prob = counts / counts.sum(-1, keepdim=True)
            loss2 = -prob[torch.arange(BS*SL), targets].log().mean()

        return logits, loss


    def generate(self, input, max=500):
        for _ in range(max):
            input = input[:, -context:]
            logits, _ = self(input)

            # Since forward flattens logits during testing with targets,
            # we make sure we pluck the very last token's predictions cleanly here.
            # If logits came back 2D from a targetless forward pass, we handle it:
            if len(logits.shape) == 3:
                logits = logits[:, -1, :]
            else:
                logits = logits.view(input.shape[0], -1, vocab_size)[:, -1, :]

            probs = F.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
            input = torch.cat((input, next_token), dim=1)
        return input




In [13]:
class Block(nn.Module):
    def __init__(self, n_heads):
        super().__init__()
        head_size = embed_size // n_heads
        self.ma = MultiHead(n_heads, head_size)
        self.feed_forward = ForwardLayer(embed_size)
        self.ln1 = nn.LayerNorm(embed_size)
        self.ln2 = nn.LayerNorm(embed_size)


    def forward(self, x):
        x = x + self.ma(self.ln1(x))
        x = x + self.feed_forward(self.ln2(x))
        return x

In [14]:
class ForwardLayer(nn.Module):
    def __init__(self, embed_size):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(embed_size, 6 * embed_size, bias=BIAS),
            nn.GELU(),
            nn.Linear(6 * embed_size, embed_size, bias=BIAS),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.network(x)

In [15]:
class MultiHead(nn.Module):
    def __init__(self, n_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(n_heads)])
        self.combine = nn.Linear(n_heads * head_size, embed_size, bias=BIAS)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x = torch.cat([head(x) for head in self.heads], dim=-1)
        x = self.combine(x)
        x = self.dropout(x)
        return x

In [16]:
class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.keys = nn.Linear(embed_size, head_size, bias=BIAS)
        self.queries = nn.Linear(embed_size, head_size, bias=BIAS)
        self.values = nn.Linear(embed_size, head_size, bias=BIAS)

        self.register_buffer('tril', torch.tril(torch.ones(context, context)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        BS, SL, Vs = x.shape
        k = self.keys(x)
        q = self.queries(x)
        v = self.values(x)

        attn_w = q @ k.transpose(-2, -1) * k.shape[-1] ** -0.5
        attn_w = attn_w.masked_fill(self.tril[:SL, :SL] == 0, float('-inf'))
        attn_w = F.softmax(attn_w, dim=-1)
        x = attn_w @ v

        return x

In [ ]:
head_size = embed_size // n_heads

In [18]:
# Training Setup

model = GPT()
model = model.to(device)
model = model.to(device)

if compile:
  print("Compiling the model...")
  model = torch.compile(model)

print(sum(p.numel() for p in model.parameters())/1e6, 'Million parameters')





19.837954 Million parameters


In [19]:
# --- TESTING AND EXECUTION ---

# x, y = get_batch("train")
# model = GPT()
# model = model.to(device)

# logit, loss = model(x, y)

# print(f"loss: {loss.item()}")

@torch.no_grad()
def generate_sample(prompt_text):
    t1 = torch.tensor(encode(prompt_text), dtype=torch.long, device=device)
    t1 = t1[None, :]  # Add batch dimension -> shape (1, sequence_length)
    newgen = model.generate(t1, max=64)[0].tolist()
    result = decode(newgen)
    print(result)

# Test it out!
# generate_sample("Once upon a time")

In [20]:
@torch.no_grad()
def calculate_loss():
    out = {}
    model.eval()
    for split in ['train', 'eval']:
        l = torch.zeros(eval_iters)
        for i in range(eval_iters):
            x, y = get_batch(split)  # Get a new batch of data
            _, loss = model(x, y)    # Calculate the loss
            l[i] = loss              # Store the loss in the next position of tensor
        out[split] = l.mean().item() # Calculate the mean and extract the final value

    model.train()  # Reset the model back to training mode after checking both splits
    return out     # Return the final dictionary

l = calculate_loss()
print(l)

{'train': 8.385480880737305, 'eval': 8.382111549377441}


In [21]:
p_dict = {p_name: p for p_name, p in model.named_parameters() if p.requires_grad}


weight_decay_p = [p for n, p in p_dict.items() if p.dim() >= 2]
no_weight_decay_p = [p for n, p in p_dict.items() if p.dim() < 2]


optimizer_groups = [
    {'params': weight_decay_p, 'weight_decay': weight_decay},
    {'params': no_weight_decay_p, 'weight_decay': 0.0}
]

optimizer = torch.optim.AdamW(optimizer_groups, lr=lr, betas=(0.9, 0.95))

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, train_iters, eta_min=lr/10)

start_iteration = 0
best_val_loss = float('inf')







In [22]:
def load_checkpoint(path):
    print("LLM - loading model")
    checkpoint = torch.load(path)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    iteration = checkpoint['iteration']
    loss = checkpoint['loss']
    print(f"Loaded iter {iteration} with loss {loss}")
    return iteration, loss

if os.path.exists(f"{checkpoint_dir}/{checkpoint_load_fn}") and load_pretrained:
    start_iteration, loss = load_checkpoint(checkpoint_dir + checkpoint_load_fn)
    best_val_loss = loss

In [23]:
if inference == True:
  model.eval()
  while True:
    qs = input("Enter text (q to quit): ")
    if qs == "":
      continue
    if qs == "q":
      break
    generate_sample(qs)



In [24]:
# =================================================================
# ====================== TRAINING LOOP ============================
# =================================================================

try:
    for i in tqdm(range(start_iteration, train_iters)):
        xb, yb = get_batch("train")  # Get a new batch of data
        logits, loss = model(xb, yb)  # Run the LLM and get the logits and the loss

        # Evaluation, Preview, and Logging Interval
        if (i % eval_interval == 0 or i == train_iters - 1):
            l = calculate_loss()
            print(f"\n{i}: train loss: {l['train']:.4f} / val loss: {l['eval']:.4f}")

            # Observe the model's text generation evolution through training
            generate_sample("The mountain in my city is")

            # Checkpoint Saving: Triggered only if validation loss improves
            if l['eval'] < best_val_loss:
                best_val_loss = l['eval']
                print("[CHECKPOINT]: Saving with loss: ", best_val_loss)

                # Safe path formatting to prevent concatenated directory string errors
                torch.save({
                    'model_state_dict': model.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(),
                    'loss': best_val_loss,
                    'iteration': i,
                }, checkpoint_dir + checkpoint_fn)

            # Weights & Biases Logging: Aligned to log EVERY interval continuously
            if wandb_log:
                wandb.log({
                    "loss/train": l['train'],
                    "loss/val": l['eval'],
                    "lr": scheduler.get_last_lr()[0],
                }, step=i)

        # Backpropagation & Optimization steps
        optimizer.zero_grad(set_to_none=True)  # Reset gradients cleanly
        loss.backward()                        # Calculate new gradients

        # Clip gradients to prevent the exploding gradient problem
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=grad_clip)

        optimizer.step()  # Update the model parameters
        scheduler.step()  # Update the learning rate via the Cosine scheduler

    if wandb_log:
        wandb.finish()

except KeyboardInterrupt:
    print("\nTraining interrupted by user. Cleaning up...")

finally:
    # Safely release and flush GPU VRAM memory allocations
    torch.cuda.empty_cache()
    print("GPU memory released.")

# Final fail-safe cleanup
if wandb_log:
    wandb.finish()
torch.cuda.empty_cache()

  0%|          | 0/100000 [00:00<?, ?it/s]


0: train loss: 8.3889 / val loss: 8.3825
The mountain in my city is Nofess Carolina schools sisterript meantards help sm place Strappen weap mustmonied oldestepba rulourn Olympics Rockuar- Later scored fieldondonattle eat and Je appear On ed inter WWE agre diseaseots cult Twyer shot inhabitup- September everII Sant education call Pe sout fish move count of profess writing
[CHECKPOINT]: Saving with loss:  8.382495880126953


  0%|          | 50/100000 [00:16<7:30:45,  3.70it/s]


50: train loss: 6.0866 / val loss: 5.9814
The mountain in my city is� be causeiber who, in hisream Vano material, and " wereican performay starting a Brazilian, and M.

Doussovernor.atero time cityn�rict wereshing important Mo that c that on theib1 It allowed, pl Sur was an'rist
[CHECKPOINT]: Saving with loss:  5.981438636779785


  0%|          | 100/100000 [00:35<8:11:36,  3.39it/s]

...

 13%|█▎        | 12551/100000 [1:14:17<30:19:47,  1.25s/it]

The mountain in my city is on earth (the way that he can see Grande a tribe about Apolli or attached paper that orbits. Much minister Manoctav College Pán Páno dawn and Debraín Cavámigán


 13%|█▎        | 12600/100000 [1:14:31<8:36:58,  2.82it/s]


Training interrupted by user. Cleaning up...
GPU memory released.


loss/train,█▇▇▆▆▅▅▅▅▄▄▄▄▄▃▄▃▃▃▃▃▃▃▃▃▂▂▂▂▂▂▁▂▂▂▁▁▁▁▁
loss/val,█▆▆▅▅▄▄▄▄▄▃▃▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂▂▁▂▁▁▁▁▂▁▁▁
lr,██████████▇▇▇▇▇▆▆▅▅▅▅▅▅▅▅▄▄▄▄▄▃▃▃▃▃▂▂▁▁▁
loss/train,2.78924
loss/val,2.98181
lr,0.00029


In [25]:
!nvidia-smi

Wed Jun  3 21:44:56 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   73C    P0             34W /   70W |    2035MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----